# Create and verify AgentReach Test Document

This notebook creates the single test Google Doc requested for AgentReach Phase 3. It searches only for the exact title `AgentReach Test Document`, and does not list any other Drive files.

**Account:** When Google shows its account chooser, select `thec7135@gmail.com`. If the notebook is already authorized to a different account and no account chooser appears, stop and switch to the correct Google account before running the creation cell. Never paste your password or verification codes into the notebook or chat.

**Title:** `AgentReach Test Document`

**Text:** `This is a non-sensitive test document for AgentReach AI. The integration is working.`

This notebook is designed to avoid duplicates and avoid overwriting: if one exact-title document exists with the correct text, it simply verifies it; if the title exists with different content or multiple exact-title documents exist, it makes no changes and tells you why. If no exact-title document exists, it creates it and verifies the result. No other files are searched or listed.

In [ ]:
%pip install -q google-api-python-client

## 2. Sign in and prepare Google API clients

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from google.auth.transport.requests import AuthorizedSession
import google.auth
import json

EXPECTED_EMAIL = 'thec7135@gmail.com'
auth.authenticate_user()
credentials, _ = google.auth.default()
session = AuthorizedSession(credentials)
profile_response = session.get('https://www.googleapis.com/oauth2/v2/userinfo', timeout=15)
profile_response.raise_for_status()
active_email = profile_response.json().get('email', '').strip().lower()
print('Signed-in Google account:', active_email or '(email unavailable)')
if active_email != EXPECTED_EMAIL:
    raise RuntimeError('Wrong or unverified Google account. No document will be created. Sign in to the requested account and rerun this cell.')

drive_service = build('drive', 'v3', credentials=credentials)
docs_service = build('docs', 'v1', credentials=credentials)
print('Account verified. Drive and Docs API clients initialized.')

## 3. Create the exact test document, without overwriting

Run this cell once. It checks only the exact test title and creates the document only if none exists.

In [ ]:
assert active_email.lower() == EXPECTED_EMAIL.lower(), 'Wrong or unverified Google account. No document will be created.'

TITLE = 'AgentReach Test Document'
EXPECTED_TEXT = 'This is a non-sensitive test document for AgentReach AI. The integration is working.'
GOOGLE_DOCS_MIME = 'application/vnd.google-apps.document'

# Only look for files with the exact title and Google Docs type. Never list unrelated files.
query = f"name = '{TITLE}' and mimeType = '{GOOGLE_DOCS_MIME}' and trashed = false"
existing_response = drive_service.files().list(
    q=query,
    pageSize=10,
    fields='files(id,name,mimeType,webViewLink,modifiedTime)',
).execute()
existing = existing_response.get('files', [])

def read_doc_text(document_id):
    doc = docs_service.documents().get(documentId=document_id).execute()
    chunks = []
    for block in doc.get('body', {}).get('content', []):
        paragraph = block.get('paragraph')
        if paragraph:
            for element in paragraph.get('elements', []):
                text_run = element.get('textRun')
                if text_run:
                    chunks.append(text_run.get('content', ''))
    return ''.join(chunks).strip()

if len(existing) > 1:
    print(f'Found {len(existing)} Google Docs with the exact test title. No files were changed to avoid ambiguity.')
    print('Please keep one test document and rename any duplicate test copies yourself, then rerun this cell.')
elif len(existing) == 1:
    candidate = existing[0]
    current_text = read_doc_text(candidate['id'])
    if current_text == EXPECTED_TEXT:
        print('The exact test document already exists and its text matches. No duplicate was created.')
        print('Title:', candidate['name'])
        print('Verified text:', current_text)
        print('Document URL:', candidate.get('webViewLink') or f"https://docs.google.com/document/d/{candidate['id']}/edit")
    else:
        print('An exact-title document already exists, but its content differs from the expected test text. No changes were made.')
        print('To avoid overwriting existing content, review that test document manually before rerunning.')
else:
    created = drive_service.files().create(
        body={'name': TITLE, 'mimeType': GOOGLE_DOCS_MIME},
        fields='id,name,mimeType,webViewLink',
    ).execute()
    try:
        docs_service.documents().batchUpdate(
            documentId=created['id'],
            body={'requests': [{'insertText': {'location': {'index': 1}, 'text': EXPECTED_TEXT}}]},
        ).execute()
        actual_text = read_doc_text(created['id'])
        print('Created Google Doc.')
        print('Title:', created['name'])
        print('Content verified:', actual_text == EXPECTED_TEXT)
        print('Verified text:', actual_text)
        print('Document URL:', created.get('webViewLink') or f"https://docs.google.com/document/d/{created['id']}/edit")
        if actual_text != EXPECTED_TEXT:
            print('Warning: the saved text differs from the expected text. Inspect the document before continuing.')
    except Exception as exc:
        print('The document was created, but inserting or verifying the text failed:', type(exc).__name__)
        print('Open this document and inspect it before retrying:')
        print(created.get('webViewLink') or f"https://docs.google.com/document/d/{created['id']}/edit")

## 4. Next step

When the output says `Content verified: True` or that the existing document text matches, open the printed document URL to confirm it. Then use the Phase 3 Safe Test notebook and search for the exact title. Never share authorization tokens or private document information in chat.